# Week 7: A Chatbot for a Domain of My Own
### AITCD001 Chatbots Development · Learning Outcome 2 · United African Institute of Technology

Everything from Weeks 3 to 6 in one chatbot, for a domain you choose. This notebook becomes the
**Portfolio Project**: the practical part of Formative Assessment 1.

**How to use it**

1. In Colab: File, then **Save a copy in Drive**. Work in the copy, and rename it after your
   chatbot, for example `clinic_chatbot.ipynb`.
2. Go step by step, from the top. Each step is marked:
   * **YOUR DOMAIN**: the cell holds an example. Replace it with your own.
   * **PROVIDED**: the machinery from Weeks 3 to 6. Read it, run it, no change needed.
3. After changing a cell, run it, then run every cell below it again (Runtime, then Run after).
4. The worked example is a made-up bus ticket desk, **City Bus Desk**, with invented fares. It
   is there to show what each step looks like. It may not be submitted as your project.

**Step 6 adds two rules to the table.** If you run it twice, run the whole notebook again from
the top (Runtime, then Restart and run all).

---

---

## Setup

**PROVIDED**: read it and run it.

- Run it.

In [1]:
# AITCD001, Week 7: a rule-based chatbot for a domain of my own
import re
from datetime import datetime

print("ready")

ready


---

## Step 1: Your domain and its users

**YOUR DOMAIN**: replace the example.

- `BOT_NAME`: the name of your chatbot.
- `USERS`: who will use it. A real group of people, as specific as possible.
- `TOPICS`: what it can help with, in a short phrase. It is used in the greeting and the fallback, so it must read well in "I can help with ...".

In [2]:
# STEP 1. YOUR DOMAIN: replace these three lines with your own.
BOT_NAME = "Clinic Appointment Assistant"
USERS = "patients of a clinic in Kigali"
TOPICS = "opening hours, services, prices, location and booking a visit"

print(f"{BOT_NAME}: a chatbot for {USERS}.")
print(f"It can help with {TOPICS}.")

Clinic Appointment Assistant: a chatbot for patients of a clinic in Kigali.
It can help with opening hours, services, prices, location and booking a visit.


---

## Step 2: Intents and their patterns

**YOUR DOMAIN**: replace the example.

- At least **five** intents for your domain, each a `(name, pattern)` pair.
- Build each pattern from real phrasings: at least three ways of asking, including one in Kinyarwanda or SMS style if your users would write that way.
- Order matters: the first rule that matches wins. Put `cancel` above what it cancels, and specific intents above general ones.
- Keep `\b` around words, as in Week 4, so that `order` does not match `border`.

In [3]:
# STEP 2. YOUR DOMAIN: your intents, in priority order. At least five.
# Each rule is (intent, pattern). The first rule that matches a request wins, so put
# "cancel" above what it cancels, and the specific above the general.
# Do not add "name" or "greeting": Step 6 adds them.
# normalise() removes punctuation, so "can't" becomes "cant".
RULES = [
    ("emergency", r"\bemergency\b|\bbleeding\b|\bcan ?t breathe\b|\bamaraso\b|\bihutirwa\b"),
    ("cancel",    r"\bcanc\w*\b|\bhagarika\w*\b|\bcall it off\b"),
    ("hours",     r"\bhours?\b|\bopen\w*\b|\bclos\w*\b|\bwhat ?time\b|\bwat ?tym\b|\bsaa ngapi\b|\bamasaha\b|\bgufungura\b"),
    ("prices",    r"\bprices?\b|\bcosts?\b|\bfees?\b|\bhow ?much\b|\bhw ?much\b|\bangahe\b|\bibiciro\b"),
    ("services",  r"\bservices?\b|\bserivisi\b|\boffer\w*\b|\bmwivura\b|\bmufite\b"),
    ("location",  r"\bwhere\b|\blocat\w*\b|\baddress\b|\bmuri he\b|\bmuherereye\b"),
    ("book",      r"\bbook\w*\b|\bappointments?\b|\bvisit\b|\breserv\w*\b|\bgahunda\b|\bkwiyandikisha\b|\bkwivuza\b"),
]

print(len(RULES), "intents in the rule table")

7 intents in the rule table


---

## Step 3: Matching and routing

**PROVIDED**: read it and run it.

- Run it. The last line routes a two request message; with your own rules, change that message to one of yours.

In [4]:
# STEP 3. PROVIDED: cleaning, matching, and several requests in one message (Weeks 3 to 5).
def normalise(text):
    """Remove punctuation (keeping the colon of 21:00), lowercase, trim."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


def match(message):
    """The tag of the first rule that matches, or "fallback"."""
    text = normalise(message)
    for tag, pattern in RULES:
        if re.search(pattern, text):
            return tag
    return "fallback"


CONNECTORS = r"\band\b|\bbut\b|\balso\b|\bthen\b|[?]"


def split_requests(message):
    """Break one message into the separate requests it contains."""
    requests = []
    for part in re.split(CONNECTORS, message.lower()):
        part = normalise(part)
        if part != "":
            requests.append(part)
    return requests


def route(message):
    """One (intent, request) pair for each request understood."""
    found = []
    seen = []
    for request in split_requests(message):
        tag = match(request)
        if tag != "fallback" and tag not in seen:
            seen.append(tag)
            found.append((tag, request))
    return found


print(route("what time do you open and how much is dental"))

[('hours', 'what time do you open'), ('prices', 'how much is dental')]


---

## Step 4: Handlers and the router

**YOUR DOMAIN**: replace the example.

- One handler function per intent in Step 2, with the **same name** as its tag in `HANDLERS`.
- At least one handler should **read the message** and work its answer out, as `handle_fares` does with a dictionary. Check with `in` before looking a key up, or build the pattern from the dictionary's keys, so no `KeyError` is possible.
- The handler that starts your conversation calls `start_flow()`; `cancel` calls `reset_flow()`. List the intents whose reply is a question in `ASKS`.
- The loop at the end must print nothing but the count: every intent needs a handler.

In [5]:
# STEP 4. YOUR DOMAIN: one handler per intent, then the router.
# NOTE: these clinic details are invented. Replace them with your real ones.
PHONE = "07XXXXXXXX"
ADDRESS = "KG 567 St, Kacyiru, Kigali"

PRICES = {"consultation": "5,000 RWF", "dental": "10,000 RWF", "lab": "7,000 RWF"}
# Other words users type, and the service each one means (English and Kinyarwanda).
ALIASES = {"consultation": "consultation", "general": "consultation",
           "dental": "dental", "dentist": "dental", "amenyo": "dental",
           "lab": "lab", "laboratory": "lab", "laboratwari": "lab"}
SERVICE_WORDS = r"\b(" + "|".join(ALIASES) + r")\b"


def handle_emergency(text):
    reset_flow()
    return f"This sounds urgent. Please call {PHONE} or go to the nearest hospital now."


def handle_hours(text):
    return "We are open Monday to Friday 08:00 to 17:00, and Saturday 08:00 to 13:00. We are closed on Sunday."


def handle_prices(text):
    """Reads the message: the price of the service named, if it is one we offer."""
    found = re.search(SERVICE_WORDS, text)
    if found:
        service = ALIASES[found.group(1)]
        return f"{service.capitalize()} costs {PRICES[service]}."
    return "Consultation 5,000 RWF, dental 10,000 RWF, lab test 7,000 RWF."


def handle_services(text):
    return "We offer general consultation, dental care and lab tests."


def handle_location(text):
    return f"We are at {ADDRESS}. You can call us on {PHONE}."


def handle_book(text):
    """Starts the booking conversation (Step 5)."""
    if flow["step"] is not None:
        return "Your booking is already under way."
    return start_flow()


def handle_cancel(text):
    if flow["step"] is not None:
        reset_flow()
        return "Your booking has been cancelled. Nothing was booked."
    return "There is no booking in progress to cancel."


HANDLERS = {
    "emergency": handle_emergency,
    "cancel":    handle_cancel,
    "hours":     handle_hours,
    "prices":    handle_prices,
    "services":  handle_services,
    "location":  handle_location,
    "book":      handle_book,
}
ASKS = ["book"]       # intents whose reply is a question: it goes last

for tag, pattern in RULES:
    if tag not in HANDLERS:
        print("No handler for:", tag)
print(len(HANDLERS), "intents routed")

7 intents routed


---

## Step 5a: Your one conversation

**YOUR DOMAIN**: replace the example.

- The questions your conversation asks, in order, each with the name of the answer and the pattern of a valid answer.
- Two to four questions is right. Make each pattern catch the answer, not the whole sentence, so the summary reads well.
- Change `DONE` and `NOT_DONE` to suit what your conversation does (booked, registered, reserved...).

In [6]:
# STEP 5a. YOUR DOMAIN: the questions your ONE conversation asks, in order.
# Each step is (what to remember, the question, the pattern of a valid answer).
FLOW_STEPS = [
    ("service", "Which service? Consultation, dental or lab test?", SERVICE_WORDS),
    ("day",     "Which day? Monday to Saturday. We are closed on Sunday.",
     r"\b(monday|tuesday|wednesday|thursday|friday|saturday|today|tomorrow)\b"),
    ("time",    "Morning or afternoon?", r"\b(morning|afternoon)\b"),
    ("phone",   "What is your phone number? For example 07XXXXXXXX.", r"\b07\d{2} ?\d{3} ?\d{3}\b"),
]
DONE = "Your appointment request is submitted. The receptionist will call you to confirm: "
NOT_DONE = "No problem. Nothing has been booked."

print(len(FLOW_STEPS), "questions, then a confirmation")

4 questions, then a confirmation


---

## Step 5b: The flow machinery

**PROVIDED**: read it and run it.

- Run it. It asks your questions in order, re-asks when an answer is not valid, and ends with **a confirmation**: nothing is done until the user says yes (yego).

In [7]:
# STEP 5b. PROVIDED: the flow (Weeks 4 and 5), now driven by FLOW_STEPS,
# and ending with a real confirmation: nothing is done until the user says yes.
flow = {"step": None, "collected": {}}
YES = r"\b(yes|yeah|yego|correct|ok|okay)\b"     # yego: yes
NO = r"\b(no|nope|oya|wrong)\b"                  # oya: no


def start_flow():
    flow["step"] = 0
    flow["collected"] = {}
    return FLOW_STEPS[0][1]


def reset_flow():
    flow["step"] = None
    flow["collected"] = {}


def summary():
    return ", ".join(flow["collected"].values())


def continue_flow(message):
    """One answer: store it, ask the next question, and at the end, confirm."""
    text = normalise(message)
    if flow["step"] == "confirm":
        if re.search(YES, text):
            booked = summary()
            reset_flow()
            return DONE + booked + ". Thank you."
        if re.search(NO, text):
            reset_flow()
            return NOT_DONE
        return "Please answer yes or no."
    name, question, pattern = FLOW_STEPS[flow["step"]]
    found = re.search(pattern, text)
    if not found:
        return question                          # not a valid answer: ask again
    flow["collected"][name] = found.group()
    flow["step"] = flow["step"] + 1
    if flow["step"] < len(FLOW_STEPS):
        return FLOW_STEPS[flow["step"]][1]       # the next question
    flow["step"] = "confirm"
    return "So: " + summary() + ". Shall I book it? Yes or no."


print("Bot:", start_flow())
for answer in ["dental please", "monday", "in the morning", "0788678615", "yes"]:
    print("You:", answer)
    print("Bot:", continue_flow(answer))

Bot: Which service? Consultation, dental or lab test?
You: dental please
Bot: Which day? Monday to Saturday. We are closed on Sunday.
You: monday
Bot: Morning or afternoon?
You: in the morning
Bot: What is your phone number? For example 07XXXXXXXX.
You: 0788678615
Bot: So: dental, monday, morning, 0788678615. Shall I book it? Yes or no.
You: yes
Bot: Your appointment request is submitted. The receptionist will call you to confirm: dental, monday, morning, 0788678615. Thank you.


---

## Step 6: Context, a name, a greeting, follow ups

**PROVIDED**: read it and run it.

- Run it **once**. It adds the name and greeting intents.
- `FOLLOW_UPS`: if one of your intents can be continued by a message such as "and musanze?", put the intent and the pattern of what continues it here. If none can, write `FOLLOW_UPS = {}`.

In [8]:
# STEP 6. PROVIDED, except FOLLOW_UPS: memory, a name, a greeting, follow ups (Week 5).
# Run this cell ONCE: it adds the name and greeting rules to RULES.
context = {"name": None, "last_intent": None, "fallbacks": 0}


def reset_context():
    context["name"] = None
    context["last_intent"] = None
    context["fallbacks"] = 0


NAME = r"\bmy name is (\w+)"


def handle_name(text):
    found = re.search(NAME, text)
    if found:
        context["name"] = found.group(1).capitalize()
        return f"Nice to meet you, {context['name']}."
    return "Sorry, I did not catch your name."


def handle_greeting(text):
    if context["name"] is not None:
        return f"Muraho, {context['name']}! How can I help?"
    return (f"Muraho! I am the {BOT_NAME}, a computer program. "
            f"I can help with {TOPICS}.")


RULES.insert(0, ("name", r"\bmy name is\b"))
RULES.append(("greeting", r"\bhello\b|\bhi\b|\bhey\b|\bmuraho\b|\bmwaramutse\b|\bmwiriwe\b"))
HANDLERS["name"] = handle_name
HANDLERS["greeting"] = handle_greeting

# YOUR DOMAIN: an intent, and the pattern that continues it ("and lab?" after a price).
FOLLOW_UPS = {"prices": SERVICE_WORDS}


def follow_up(text):
    """A message with no intent of its own, read in the light of the last one."""
    last = context["last_intent"]
    if last in FOLLOW_UPS and re.search(FOLLOW_UPS[last], text):
        return [(last, text)]
    return []


print(len(RULES), "rules,", len(HANDLERS), "intents routed")

9 rules, 9 intents routed


---

## Step 7: Validation, errors, the fallback and a person

**PROVIDED**: read it and run it.

- Run it. The fallback and the request to type a question use your `TOPICS`.

In [9]:
# STEP 7. PROVIDED: validation, errors, the fallback and a person (Week 6).
MAX_LENGTH = 300
ASK_AGAIN = f"Please type a question. I can help with {TOPICS}."
IN_WORDS = "Please type your question in words."
TOO_LONG = "That is a long message. Could you ask one thing at a time?"


def validate(message):
    """Return a reply if the message cannot be used; None if it can."""
    if message is None:
        return ASK_AGAIN
    text = message.strip()
    if text == "":
        return ASK_AGAIN
    if normalise(text) == "":
        return IN_WORDS
    if len(text) > MAX_LENGTH:
        return TOO_LONG
    return None


ERROR_LOG = []
SORRY = "Sorry, something went wrong on my side. Please try again, or type 'human'."


def log_error(message, error):
    ERROR_LOG.append({"time": datetime.now().strftime("%H:%M:%S"), "message": message,
                      "error": type(error).__name__ + ": " + str(error)})


def safe_handle(tag, request):
    """Run one handler. If it raises an error, log it and apologise instead of crashing."""
    try:
        return HANDLERS[tag](request)
    except Exception as error:
        log_error(request, error)
        return SORRY


def respond(found):
    """Answer every request found; a reply that asks a question goes last."""
    answers = []
    questions = []
    for tag, request in found:
        reply = safe_handle(tag, request)
        if tag in ASKS:
            questions.append(reply)
        else:
            answers.append(reply)
    return " ".join(answers + questions)


UNMATCHED = []
FALLBACK = f"Sorry, I did not understand. I can help with {TOPICS}."
STILL_LOST = f"Sorry, I still did not understand. Type 'human' for a member of staff, or ask about {TOPICS}."


def fallback(message):
    UNMATCHED.append(message)
    context["fallbacks"] = context["fallbacks"] + 1
    if context["fallbacks"] >= 2:
        return STILL_LOST
    return FALLBACK


EXIT_WORDS = ("bye", "goodbye", "exit", "quit", "thanks", "murakoze")
HUMAN_WORDS = ("human", "staff", "agent", "person")
HANDOVER = "Connecting you to a member of staff. They can see this conversation."

print("validation, error handling and the fallback are ready")

validation, error handling and the fallback are ready


---

## Step 7, continued: the complete turn

**PROVIDED**: read it and run it.

- Run it. The demonstration at the end uses the example; change it to a conversation of your own.

In [10]:
# STEP 7, continued. PROVIDED: one complete turn, in the Week 6 order.
def reply_to(message):
    problem = validate(message)                  # 0. unusable input
    if problem is not None:
        return problem
    text = normalise(message)

    if text in EXIT_WORDS:                       # 1. the exit, at any point
        name = context["name"]
        reset_flow()
        reset_context()
        if name is not None:
            return f"Thank you, {name}. Murakoze."
        return "Thank you for contacting us. Murakoze."

    if text in HUMAN_WORDS:                      # 2. a person, at any point
        reset_flow()
        return HANDOVER

    if flow["step"] is not None:                 # 3. the flow sees it first
        step_before = flow["step"]
        flow_reply = continue_flow(message)
        if flow["step"] != step_before:
            return flow_reply
        found = route(message)
        if len(found) == 0:
            return flow_reply
        reply = respond(found)
        if flow["step"] is not None:
            reply = reply + " " + flow_reply
        return reply

    found = route(message)                       # 4. route
    if len(found) == 0:
        found = follow_up(text)                  # 5. a follow up
    if len(found) == 0:
        return fallback(message)                 # 6. the fallback, counted

    context["fallbacks"] = 0
    last_tag, last_request = found[-1]
    context["last_intent"] = last_tag
    return respond(found)


def run_conversation(turns):
    """Play a conversation from a clean start, printing both sides."""
    reset_flow()
    reset_context()
    for message in turns:
        print("You:", message)
        print("Bot:", reply_to(message))
    print()


run_conversation(["muraho", "how much is dental", "and lab?", "bye"])

You: muraho
Bot: Muraho! I am the Clinic Appointment Assistant, a computer program. I can help with opening hours, services, prices, location and booking a visit.
You: how much is dental
Bot: Dental costs 10,000 RWF.
You: and lab?
Bot: Lab costs 7,000 RWF.
You: bye
Bot: Thank you for contacting us. Murakoze.



---

## Step 8: Ten test conversations

**YOUR DOMAIN**: replace the example.

- Write ten conversations for **your** chatbot. Write what you expect **on paper first**.
- Include: two requests in one message, the whole conversation confirmed, the conversation refused or cancelled, a side question, a name, local phrasing, unexpected input, and one you think will break the chatbot.
- A failed test is not a lost mark. It is what goes into the README.

In [11]:
# STEP 8. YOUR DOMAIN: ten test conversations. Expected results written BEFORE running.
# Test 1  Greeting says it is a program; name is remembered; goodbye uses the name.
# Test 2  Hours AND dental price answered in one reply.
# Test 3  Four questions, summary, "yes" -> request submitted with the details.
# Test 4  Four questions, summary, "no" -> nothing booked.
# Test 5  "where are you" mid-booking: the address, then the day question is asked again.
# Test 6  "cancel" mid-booking: cancelled, nothing booked.
# Test 7  "and lab?" after a dental price -> the lab price (follow up).
# Test 8  Empty input, two unknown messages (second fallback offers a person), then "human".
# Test 9  Kinyarwanda hours, "amenyo ni angahe" -> dental price, then "ihutirwa" -> emergency.
# Test 10 (may break) "book a dental visit": should skip the service question and ask the day.
TESTS = [
    ('A greeting, a name, a goodbye',
     ['muraho', 'my name is aline', 'bye']),
    ('Two requests in one message',
     ['what time do you open and how much is dental']),
    ('A booking, confirmed',
     ['i want to book a visit', 'dental', 'monday', 'morning', '0788123456', 'yes']),
    ('A booking, refused at the end',
     ['book an appointment', 'consultation', 'tomorrow', 'afternoon', '0788 123 456', 'no']),
    ('A side question during the booking',
     ['book a visit', 'lab', 'where are you', 'tuesday']),
    ('Cancelling halfway',
     ['book a visit', 'dental', 'cancel']),
    ('A follow up',
     ['how much is dental', 'and lab?']),
    ('Unexpected input, then a person',
     ['', 'asdfgh', 'qwerty', 'human']),
    ('Local phrasing and an emergency',
     ['mufungura saa ngapi', 'amenyo ni angahe', 'ihutirwa']),
    ('A booking with the service already given',
     ['book a dental visit']),
]

number = 1
for title, turns in TESTS:
    print(f"Test {number}: {title}")
    run_conversation(turns)
    number = number + 1

Test 1: A greeting, a name, a goodbye
You: muraho
Bot: Muraho! I am the Clinic Appointment Assistant, a computer program. I can help with opening hours, services, prices, location and booking a visit.
You: my name is aline
Bot: Nice to meet you, Aline.
You: bye
Bot: Thank you, Aline. Murakoze.

Test 2: Two requests in one message
You: what time do you open and how much is dental
Bot: We are open Monday to Friday 08:00 to 17:00, and Saturday 08:00 to 13:00. We are closed on Sunday. Dental costs 10,000 RWF.

Test 3: A booking, confirmed
You: i want to book a visit
Bot: Which service? Consultation, dental or lab test?
You: dental
Bot: Which day? Monday to Saturday. We are closed on Sunday.
You: monday
Bot: Morning or afternoon?
You: morning
Bot: What is your phone number? For example 07XXXXXXXX.
You: 0788123456
Bot: So: dental, monday, morning, 0788123456. Shall I book it? Yes or no.
You: yes
Bot: Your appointment request is submitted. The receptionist will call you to confirm: dental, mo

---

## Step 9: Refine, and test again

**YOUR DOMAIN**: replace the example.

- Choose one test that failed. Change the code that caused it, then run that test again.
- The example: test 10 failed because the booking handler ignored the town already in the message. The new handler uses every answer the request already contains.
- Record it in the README, under "A bug found and fixed": what happened, the cause, the change.

In [12]:
# STEP 9. Refine: test 10 failed. The booking handler ignored the service in the request.
# The change: after starting the flow, use every answer the request already contains.
def handle_book(text):
    """Refined after test 10: answers already in the request are used."""
    if flow["step"] is not None:
        return "Your booking is already under way."
    reply = start_flow()
    while flow["step"] != "confirm":
        pattern = FLOW_STEPS[flow["step"]][2]     # the current question's pattern
        if not re.search(pattern, text):
            break                                 # the request does not answer it: ask
        reply = continue_flow(text)               # it does: answer it, and go on
    return reply


HANDLERS["book"] = handle_book      # the router keeps the old function until this line

title, turns = TESTS[9]
print("Test 10 again:", title)
run_conversation(turns)

Test 10 again: A booking with the service already given
You: book a dental visit
Bot: Which day? Monday to Saturday. We are closed on Sunday.



---

## Step 10: Validate against local phrasing

**YOUR DOMAIN**: replace the example.

- Replace `PHRASING` with at least five messages written the way **your** users write: Kinyarwanda, mixed, SMS spelling.
- Add what you can to the right rule in Step 2, run everything again, and record what is still missed under "Known limitations".

In [13]:
# STEP 10. YOUR DOMAIN: messages written the way your users really write:
# Kinyarwanda, mixed, SMS spelling. Then see which ones end up in UNMATCHED.
PHRASING = [
    'mufungura saa ngapi',
    'muraho, how much is dental',
    'wat tym u open',
    'hw much 4 dental',
    'book visit 2moro',
    'cancle',
    'ndashaka kwivuza',
    'aho muherereye',
    'nshaka umuganga',
    'pls help me',
]


def not_understood(messages):
    """Run each message from a clean start. Return the ones that ended up in UNMATCHED."""
    missed = []
    for message in messages:
        reset_flow()
        reset_context()
        before = len(UNMATCHED)
        reply_to(message)
        if len(UNMATCHED) > before:
            missed.append(message)
    reset_flow()
    reset_context()
    return missed


missed = not_understood(PHRASING)
print(len(missed), "of", len(PHRASING), "not understood:", missed)

2 of 10 not understood: ['nshaka umuganga', 'pls help me']


---

## The two logs

**PROVIDED**: read it and run it.

- What your chatbot did not understand, and what broke. Both feed the README.

In [14]:
# The two logs: evidence for the README's "Known limitations" and "A bug found and fixed".
print(len(UNMATCHED), "in UNMATCHED:", UNMATCHED)
print(len(ERROR_LOG), "in ERROR_LOG:", [e["error"] for e in ERROR_LOG])

4 in UNMATCHED: ['asdfgh', 'qwerty', 'nshaka umuganga', 'pls help me']
0 in ERROR_LOG: []


---

## Step 11: Talk to it

Run the cell below **on its own**: it waits for keyboard input. This is the conversation to take a picture of for the README (Windows key + Shift + S).

In [15]:
def chat():
    """The finished chatbot. This cell waits for keyboard input: run it on its own."""
    reset_flow()
    reset_context()
    print(f"{BOT_NAME}: an automated assistant, a computer program, not a member of staff.")
    print(f"It can help with {TOPICS}. Type 'human' for a person, or 'bye' to finish.")
    print()
    while True:
        message = input("You: ")
        print("Bot:", reply_to(message))
        print()
        if normalise(message) in EXIT_WORDS or normalise(message) in HUMAN_WORDS:
            break


chat()

Clinic Appointment Assistant: an automated assistant, a computer program, not a member of staff.
It can help with opening hours, services, prices, location and booking a visit. Type 'human' for a person, or 'bye' to finish.

You: muraho
Bot: Muraho! I am the Clinic Appointment Assistant, a computer program. I can help with opening hours, services, prices, location and booking a visit.

You: How much is dental?
Bot: Dental costs 10,000 RWF.

You: where are you?
Bot: We are at KG 567 St, Kacyiru, Kigali. You can call us on 07XXXXXXXX.

You: dental
Bot: Sorry, I did not understand. I can help with opening hours, services, prices, location and booking a visit.

You: monday
Bot: Sorry, I still did not understand. Type 'human' for a member of staff, or ask about opening hours, services, prices, location and booking a visit.

You: 0788000000
Bot: Sorry, I still did not understand. Type 'human' for a member of staff, or ask about opening hours, services, prices, location and booking a visit.

